# Manuscript Analysis — Full Nine-Group Reproduction

End-to-end reproduction of the main-text and supplementary figures and tables from the accompanying manuscript. Runs the EchoBot and EK80 pipelines on all nine comparison groups, computes cross-instrument Pearson correlations, SNR statistics (both methods), linearity diagnostics, and the EK80 calibration validation against the Simrad EK80 desktop software.

**Runtime:** ~5–10 minutes on a laptop, depending on how many EchoBot files need to be loaded (each `.mat` is ~150–220 MB).

**Prerequisites:** the full Zenodo raw data archive extracted into `../data/` — see [`docs/DATA.md`](../docs/DATA.md).

**Outputs:** every figure is written to `../output/figures/` (gitignored) and every summary statistic is written to `../output/stats/`.

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from echobot import (
    io,
    echobot_pipeline,
    ek80_pipeline,
    snr,
    compare,
    linearity,
    plotting,
    config,
)

DATA = Path('..') / 'data'
ECHOBOT_DIR = DATA / 'echobot' / '0311-CRL-tests'
EK80_DIR = DATA / 'EK80' / '0311-CRL-tests'
NOAA_FILE = DATA / 'NOAA-381-WC-TSf.xlsx'
VAL_TS120 = DATA / 'validation' / 'validation_0311_ping_ts120.csv'
VAL_TSF = DATA / 'validation' / 'validation_0311_tsf_detailed.csv'

OUT_FIG = Path('..') / 'output' / 'figures'
OUT_STATS = Path('..') / 'output' / 'stats'
OUT_FIG.mkdir(parents=True, exist_ok=True)
OUT_STATS.mkdir(parents=True, exist_ok=True)

print('Data paths OK:', all(p.exists() for p in (ECHOBOT_DIR, EK80_DIR, NOAA_FILE, VAL_TS120, VAL_TSF)))

## 1. Comparison group definitions

Maps each group ID to the EchoBot and EK80 file indices in the chronologically sorted file lists. These indices match the group definitions used in the manuscript.

In [ ]:
comparisons = [
    dict(cid='A1',     name='Baseline',              eb_down_idx=0,  eb_up_idx=7,  ek_idx=0),
    dict(cid='A1-dup', name='Baseline (duplicate)',  eb_down_idx=1,  eb_up_idx=None, ek_idx=0),
    dict(cid='A1-rep', name='Baseline (late repeat)',eb_down_idx=13, eb_up_idx=12, ek_idx=0),
    dict(cid='B1',     name='Tx Duration 1.0 ms',    eb_down_idx=5,  eb_up_idx=10, ek_idx=3),
    dict(cid='C1',     name='Bandwidth 100-140',     eb_down_idx=6,  eb_up_idx=11, ek_idx=4),
    dict(cid='D1',     name='No Target',             eb_down_idx=14, eb_up_idx=15, ek_idx=7),
    dict(cid='P1-lo',  name='Low power',             eb_down_idx=3,  eb_up_idx=8,  ek_idx=0),
    dict(cid='P1-mid', name='Medium power',          eb_down_idx=13, eb_up_idx=12, ek_idx=1),
    dict(cid='P1-hi',  name='High power',            eb_down_idx=4,  eb_up_idx=9,  ek_idx=2),
]

eb_files = io.list_echobot_files(ECHOBOT_DIR)
ek_files = io.list_ek80_files(EK80_DIR)
print(f'Found {len(eb_files)} EB files, {len(ek_files)} EK80 files')

eb_needed = set()
ek_needed = set()
for g in comparisons:
    ek_needed.add(g['ek_idx'])
    eb_needed.add(g['eb_down_idx'])
    if g['eb_up_idx'] is not None:
        eb_needed.add(g['eb_up_idx'])
print(f'  need {len(eb_needed)} unique EB files and {len(ek_needed)} unique EK80 files')

## 2. Batch-process EchoBot and EK80

Run the two pipelines on every unique file required by the comparisons. Results are cached in the `eb_proc` and `ek_proc` dictionaries keyed by file index.

In [ ]:
eb_proc = {}
for idx in sorted(eb_needed):
    print(f'EB #{idx+1}: {eb_files[idx].name}', flush=True)
    run = io.load_echobot_mat(eb_files[idx])
    eb_proc[idx] = echobot_pipeline.process_echobot_run(run)
print(f'\nProcessed {len(eb_proc)} EB files')

In [ ]:
ek_proc = {}
for idx in sorted(ek_needed):
    print(f'EK80 #{idx+1}: {ek_files[idx].name}', flush=True)
    ed = io.open_ek80_raw(ek_files[idx])
    ek_proc[idx] = ek80_pipeline.process_ek80_ed(ed)
print(f'\nProcessed {len(ek_proc)} EK80 files')

## 3. Cross-instrument comparison (Table 3, Figures 4 + 5)

For every group × sweep, compute the Pearson r of mean-subtracted TS(f) spectra on the 100–140 kHz overlap band.

In [ ]:
df_r = compare.per_group_comparison(
    comparisons, eb_proc, ek_proc,
    f_lo_hz=100e3, f_hi_hz=140e3,
)
print(df_r.to_string(index=False))
df_r.to_csv(OUT_STATS / 'cross_instrument_r.csv', index=False)

In [ ]:
# Figure 4: A1 baseline comparison
a1 = next(g for g in comparisons if g['cid'] == 'A1')
fig4 = plotting.plot_baseline_comparison(
    eb_proc[a1['eb_down_idx']],
    ek_proc[a1['ek_idx']],
    title='A1 Baseline',
)
fig4.savefig(OUT_FIG / 'fig04_baseline_comparison.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: 9-group normalized TS(f) overlay
fig5, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True, sharey=True)
for ax, g in zip(axes.flat, comparisons):
    ek = ek_proc.get(g['ek_idx'])
    eb = eb_proc.get(g['eb_down_idx'])
    if ek is None or eb is None:
        ax.set_visible(False); continue
    plotting.plot_tsf_overlay(
        eb.f_hz, eb.tsf_db,
        ek.f_band_sorted, ek.tsf_calibrated_db,
        normalized=True, ax=ax,
    )
    r_row = df_r[(df_r['cid']==g['cid']) & (df_r['sweep']=='down')]
    r_val = float(r_row['pearson_r'].iloc[0]) if len(r_row) else float('nan')
    ax.set_title(f"{g['cid']}: {g['name']}  (r={r_val:.3f})", fontsize=10)
    ax.legend().set_visible(False)
fig5.suptitle('TS(f) spectral-shape preservation across parameter combinations', fontsize=12)
fig5.tight_layout()
fig5.savefig(OUT_FIG / 'fig05_spectral_preservation.png', dpi=200, bbox_inches='tight')
plt.show()

## 4. Linearity diagnostics (Figure 6, Table 6)

Three orthogonal linearity tests:
- **Power**: peak-level slope across P1-lo/mid/hi
- **Pulse duration**: TS(f) shape Pearson r between A1 and B1 (full 90–150 kHz band)
- **Bandwidth**: TS(f) shape Pearson r between A1 and C1 on the 100–140 kHz overlap band

In [ ]:
# Power linearity — EchoBot (scale 0 / 3 / 5 dB) and EK80 (60 / 90 / 120 W)
eb_power_series = [
    dict(nominal_db=0, processed=eb_proc[3]),   # P1-lo
    dict(nominal_db=3, processed=eb_proc[13]),  # P1-mid
    dict(nominal_db=5, processed=eb_proc[4]),   # P1-hi
]
ek_power_series = [
    dict(nominal_db=10*np.log10(60/60.0),  processed=ek_proc[0]),
    dict(nominal_db=10*np.log10(90/60.0),  processed=ek_proc[1]),
    dict(nominal_db=10*np.log10(120/60.0), processed=ek_proc[2]),
]
eb_power = linearity.power_linearity(eb_power_series, 'EchoBot')
ek_power = linearity.power_linearity(ek_power_series, 'EK80')
print(f'EB power linearity: slope={eb_power.slope:.3f} dB/dB, rms_resid={eb_power.rms_residual_db:.3f} dB')
print(f'EK power linearity: slope={ek_power.slope:.3f} dB/dB, rms_resid={ek_power.rms_residual_db:.3f} dB')

# Duration shape invariance (A1 vs B1)
eb_dur = linearity.duration_shape_r(eb_proc[0],  eb_proc[5], 'EchoBot')
ek_dur = linearity.duration_shape_r(ek_proc[0],  ek_proc[3], 'EK80')
print(f'EB duration r={eb_dur.pearson_r:.3f}, EK duration r={ek_dur.pearson_r:.3f}')

# Bandwidth shape invariance (A1 vs C1, normalized over 100-140 kHz)
eb_bw  = linearity.bandwidth_shape_r(eb_proc[0],  eb_proc[6], 'EchoBot')
ek_bw  = linearity.bandwidth_shape_r(ek_proc[0],  ek_proc[4], 'EK80')
print(f'EB bandwidth r={eb_bw.pearson_r:.3f}, EK bandwidth r={ek_bw.pearson_r:.3f}')

lin_summary = linearity.summarize_linearity(
    power_results=[eb_power, ek_power],
    shape_results=[eb_dur, ek_dur, eb_bw, ek_bw],
)
lin_summary.to_csv(OUT_STATS / 'linearity_summary.csv', index=False)
lin_summary

## 5. SNR analysis (Figure 7, Tables S1)

Two methods applied per group:
- `snr_no_target_ref`: dedicated D1 no-target run as noise reference
- `snr_empty_region`: within-ping 2.1–2.6 m empty-water gate

In [ ]:
# D1 no-target run is the reference for the no-target method
d1 = next(g for g in comparisons if g['cid'] == 'D1')
eb_nt = eb_proc[d1['eb_down_idx']]
ek_nt = ek_proc[d1['ek_idx']]

snr_rows = []
for g in comparisons:
    if g['cid'] == 'D1':
        continue
    eb = eb_proc.get(g['eb_down_idx'])
    ek = ek_proc.get(g['ek_idx'])
    if eb is None or ek is None:
        continue

    # Empty-region (both instruments)
    eb_snr_er = snr.snr_empty_region(eb.all_mf, eb.r_mf, eb.target_gate)
    ek_snr_er = snr.snr_empty_region(ek.all_pc, ek.r_ek, ek.target_gate)

    # No-target reference (both instruments)
    eb_snr_nt = snr.snr_no_target_ref(eb.all_mf, eb.target_gate, eb_nt.all_mf)
    ek_snr_nt = snr.snr_no_target_ref(ek.all_pc, ek.target_gate, ek_nt.all_pc)

    snr_rows.append(dict(
        cid=g['cid'], name=g['name'],
        eb_empty_mean=eb_snr_er.mean_db, eb_empty_std=eb_snr_er.std_db,
        ek_empty_mean=ek_snr_er.mean_db, ek_empty_std=ek_snr_er.std_db,
        eb_ntref_mean=eb_snr_nt.mean_db, eb_ntref_std=eb_snr_nt.std_db,
        ek_ntref_mean=ek_snr_nt.mean_db, ek_ntref_std=ek_snr_nt.std_db,
    ))

df_snr = pd.DataFrame(snr_rows)
df_snr.to_csv(OUT_STATS / 'snr_per_group.csv', index=False)
df_snr.round(2)

## 6. EK80 calibration validation (Figure S10)

Compare the pipeline's calibrated TS(f) against the Simrad EK80 desktop software using the bundled validation CSVs.

In [ ]:
df_ts120 = pd.read_csv(VAL_TS120)
df_tsf = pd.read_csv(VAL_TSF)

a1_ek = ek_proc[0]
mask_ek = (a1_ek.f_band_sorted >= 90e3) & (a1_ek.f_band_sorted <= 150e3)

# Compute pipeline TS@120 for every ping in df_ts120
df_ts120 = df_ts120.copy()
df_ts120['pipeline_cal_TS120'] = np.nan
for k, row in df_ts120.iterrows():
    p = int(row['ping_index'])
    j_lo, j_hi = a1_ek.target_gate
    spec = np.fft.fft(a1_ek.all_pc[p, j_lo:j_hi] * np.hanning(j_hi-j_lo), n=config.NFFT)
    from echobot.ek80_pipeline import compute_absolute_tsf
    tsf_cal = compute_absolute_tsf(
        spec, a1_ek.tx_power_band, a1_ek.band_idx_sorted, a1_ek.f_band_sorted,
        a1_ek.target_center_m, a1_ek.meta, a1_ek.norm_fac,
    )
    df_ts120.at[k, 'pipeline_cal_TS120'] = float(
        np.interp(120e3, a1_ek.f_band_sorted[mask_ek], tsf_cal[mask_ek])
    )

# Per-ping TS(f) for the 5 detailed pings
ping_indices_5 = sorted(df_tsf['ping_index'].unique())
per_ping_tsf = {}
freq_labels = [90, 95, 100, 110, 120, 125, 130, 135, 140, 150]
freq_actual = [90, 95, 100, 110, 120, 125, 130, 135, 140, 149.9]

rows = []
for p in ping_indices_5:
    j_lo, j_hi = a1_ek.target_gate
    spec = np.fft.fft(a1_ek.all_pc[p, j_lo:j_hi] * np.hanning(j_hi-j_lo), n=config.NFFT)
    tsf_cal = compute_absolute_tsf(
        spec, a1_ek.tx_power_band, a1_ek.band_idx_sorted, a1_ek.f_band_sorted,
        a1_ek.target_center_m, a1_ek.meta, a1_ek.norm_fac,
    )
    per_ping_tsf[p] = tsf_cal
    sub = df_tsf[df_tsf['ping_index']==p].sort_values('freq_kHz').reset_index(drop=True)
    for lbl, f_true in zip(freq_labels, freq_actual):
        sw_val = float(sub[sub['freq_kHz']==lbl]['ek80_software_TS_dB'].values[0])
        cal_val = float(np.interp(f_true*1e3, a1_ek.f_band_sorted[mask_ek], tsf_cal[mask_ek]))
        rows.append(dict(
            ping=p, freq_label=lbl, freq_actual=f_true,
            pipeline_cal=cal_val, software=sw_val,
            resid=cal_val-sw_val, is_null=(lbl==135),
        ))
df_r_val = pd.DataFrame(rows)
df_no_null = df_r_val[~df_r_val['is_null']]
print(f'Pipeline vs software residuals:')
print(f'  TS@120 mean residual: {(df_ts120["pipeline_cal_TS120"] - df_ts120["ek80_software_TS_120kHz_dB"]).mean():+.3f} dB')
print(f'  TSF mean residual (excl 135 null): {df_no_null["resid"].mean():+.3f} dB')

In [ ]:
figS10 = plotting.plot_ek80_validation(df_ts120, df_r_val, per_ping_tsf, a1_ek.f_band_sorted, mask_ek)
figS10.savefig(OUT_FIG / 'supp_ek80_validation.png', dpi=200, bbox_inches='tight')
plt.show()

## 7. Absolute TS(f) comparison (Figure S11)

In [ ]:
noaa = pd.read_excel(NOAA_FILE)
noaa.columns = [c.strip() for c in noaa.columns]
f_noaa = noaa['Frequency'].values * 1e3
ts_noaa = noaa['dB (*-1)'].values

a1_eb = eb_proc[0]
figS11 = plotting.plot_absolute_tsf(
    a1_eb.f_hz, a1_eb.tsf_db,
    a1_ek.f_band_sorted, a1_ek.tsf_calibrated_db,
    f_noaa, ts_noaa,
    title='A1 Baseline TS(f): Absolute Comparison',
)
figS11.savefig(OUT_FIG / 'supp_absolute_tsf.png', dpi=200, bbox_inches='tight')
plt.show()

## Summary

All main-text and supplementary figures + stats tables have been written to `../output/figures/` and `../output/stats/`. The analysis is fully reproducible from the raw Zenodo data via this notebook; a CLI wrapper is in [`scripts/generate_manuscript_figures.py`](../scripts/generate_manuscript_figures.py).